In [ ]:
# Cell 1: Environment Setup and Library Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Clustering & NLP Imports
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
import scipy.cluster.hierarchy as sch

# Plotting Configuration
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("All libraries imported successfully!")

In [ ]:
# Cell 2: Project 1 - Customer Segmentation Synthetic Dataset Generation
np.random.seed(42)
n_customers = 300

# Generating 3 synthetic behavior clusters
c1_age = np.random.normal(25, 5, 100)
c1_income = np.random.normal(25, 8, 100)
c1_score = np.random.normal(78, 10, 100)

c2_age = np.random.normal(42, 8, 100)
c2_income = np.random.normal(85, 12, 100)
c2_score = np.random.normal(82, 9, 100)

c3_age = np.random.normal(52, 9, 100)
c3_income = np.random.normal(80, 15, 100)
c3_score = np.random.normal(20, 8, 100)

df_customer = pd.DataFrame({
    'CustomerID': [f'CUST_{i+1001}' for i in range(n_customers)],
    'Age': np.clip(np.concatenate([c1_age, c2_age, c3_age]), 18, 70).round(),
    'Annual_Income_k$': np.clip(np.concatenate([c1_income, c2_income, c3_income]), 15, 140).round(1),
    'Spending_Score_1_100': np.clip(np.concatenate([c1_score, c2_score, c3_score]), 1, 99).round()
})

df_customer.to_csv("customer_segmentation_data.csv", index=False)
print("Project 1 Dataset Shape:", df_customer.shape)
df_customer.head()

In [ ]:
# Cell 3: Project 1 - Scaling, Elbow Method & Silhouette Analysis
features_cust = ['Age', 'Annual_Income_k$', 'Spending_Score_1_100']
X_cust = df_customer[features_cust]

# Feature Scaling
scaler_cust = StandardScaler()
X_cust_scaled = scaler_cust.fit_transform(X_cust)

# Find Optimal Clusters (Elbow Method & Silhouette Scores)
wcss = []
silhouette_scores = []
k_range = range(2, 9)

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_cust_scaled)
    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_cust_scaled, kmeans.labels_))

# Plotting Evaluation Curves
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(k_range, wcss, marker='o', color='purple')
ax[0].set_title('Elbow Method (WCSS vs K)')
ax[0].set_xlabel('Number of Clusters (K)')
ax[0].set_ylabel('WCSS (Inertia)')

ax[1].plot(k_range, silhouette_scores, marker='s', color='darkorange')
ax[1].set_title('Silhouette Score vs K')
ax[1].set_xlabel('Number of Clusters (K)')
ax[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

# Fit Optimal Model (K=3)
optimal_k = 3
kmeans_final = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df_customer['Cluster_KMeans'] = kmeans_final.fit_predict(X_cust_scaled)

print(f"K-Means (K={optimal_k}) Silhouette Score:", round(silhouette_score(X_cust_scaled, df_customer['Cluster_KMeans']), 4))
print(f"K-Means (K={optimal_k}) Davies-Bouldin Index:", round(davies_bouldin_score(X_cust_scaled, df_customer['Cluster_KMeans']), 4))

In [ ]:
# Cell 4: Project 1 - Visualizing Customer Segments & Business Profiling
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_customer, 
    x='Annual_Income_k$', 
    y='Spending_Score_1_100', 
    hue='Cluster_KMeans', 
    palette='Set1', 
    s=70, 
    style='Cluster_KMeans'
)

plt.title('Customer Segments: Annual Income vs Spending Score')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend(title='Cluster')
plt.show()

# Business Profiling Summary
cluster_profile = df_customer.groupby('Cluster_KMeans')[features_cust].mean().round(2)
print("--- Customer Cluster Profiles (Feature Averages) ---")
print(cluster_profile)

In [ ]:
# Cell 5: Project 2 - News Article Text Dataset Generation
tech_articles = [
    "Artificial intelligence and machine learning models are advancing rapidly in deep learning applications.",
    "Quantum computing and advanced semiconductor chips are driving high-performance software innovation.",
    "Cybersecurity systems use neural networks and cloud algorithms to block malware and ransomware threats.",
    "Tech companies launch new AI smartphones equipped with fast processors and automated assistant tools.",
    "Data science, Python automation, and cloud server scaling power modern software architecture."
] * 12

sports_articles = [
    "The football championship ended with a thrilling match as the striker scored the winning goal in extra time.",
    "Olympic athletes train intensely for track and field events, breaking national sprint records.",
    "Cricket league tournament finals showcase spectacular batting and fast bowling performances.",
    "Basketball teams compete in national playoffs with high-scoring defense and tactical plays.",
    "Tennis players prepare for Grand Slam tournaments on grass and clay court championships."
] * 12

finance_articles = [
    "Central banks adjust interest rates to control inflation amidst global stock market trading fluctuations.",
    "Wall Street investors track corporate earnings reports, equity markets, and venture capital funding.",
    "Cryptocurrency markets and blockchain technology face regulatory scrutiny from financial agencies.",
    "Global revenue growth, quarterly profit margins, and bond yields impact international market trade.",
    "Personal finance advisors recommend index fund diversification, stock portfolios, and real estate investments."
] * 12

# Combine into single corpus
raw_articles = tech_articles + sports_articles + finance_articles
true_categories = ['Tech']*60 + ['Sports']*60 + ['Finance']*60

df_news = pd.DataFrame({
    'Article_ID': [f'ART_{i+1:03d}' for i in range(len(raw_articles))],
    'Text': raw_articles,
    'Category_Label': true_categories
})

df_news.to_csv("news_articles_dataset.csv", index=False)
print("Project 2 Dataset Shape:", df_news.shape)
df_news.head()

In [ ]:
# Cell 6: Project 2 - TF-IDF Extraction & Hierarchical Clustering Dendrogram
tfidf = TfidfVectorizer(stop_words='english', max_features=100)
X_tfidf = tfidf.fit_transform(df_news['Text']).toarray()

print("TF-IDF Matrix Shape:", X_tfidf.shape)

# Plot Dendrogram for Hierarchical Clustering
plt.figure(figsize=(10, 5))
dendrogram = sch.dendrogram(sch.linkage(X_tfidf, method='ward'))
plt.title('Hierarchical Clustering Dendrogram (News Articles)')
plt.xlabel('Article Index')
plt.ylabel('Euclidean Distance')
plt.show()

In [ ]:
# Cell 7: Project 2 - K-Means vs Agglomerative Comparison & PCA Visuals
num_clusters = 3

# K-Means Model
kmeans_news = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df_news['KMeans_Cluster'] = kmeans_news.fit_predict(X_tfidf)

# Agglomerative Hierarchical Model
agg_news = AgglomerativeClustering(n_clusters=num_clusters, metric='euclidean', linkage='ward')
df_news['Hierarchical_Cluster'] = agg_news.fit_predict(X_tfidf)

# Evaluate Both Algorithms
km_sil = silhouette_score(X_tfidf, df_news['KMeans_Cluster'])
km_db = davies_bouldin_score(X_tfidf, df_news['KMeans_Cluster'])

agg_sil = silhouette_score(X_tfidf, df_news['Hierarchical_Cluster'])
agg_db = davies_bouldin_score(X_tfidf, df_news['Hierarchical_Cluster'])

comparison_df = pd.DataFrame({
    'Metric': ['Silhouette Score (Higher = Better)', 'Davies-Bouldin Index (Lower = Better)'],
    'K-Means': [round(km_sil, 4), round(km_db, 4)],
    'Hierarchical Clustering': [round(agg_sil, 4), round(agg_db, 4)]
})

print("--- Project 2 Model Comparison ---")
print(comparison_df.to_string(index=False))

# 2D PCA Dimensionality Reduction Plot
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_tfidf)

df_news['PCA1'] = X_pca[:, 0]
df_news['PCA2'] = X_pca[:, 1]

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_news, 
    x='PCA1', 
    y='PCA2', 
    hue='KMeans_Cluster', 
    style='Category_Label', 
    palette='viridis', 
    s=80
)

plt.title('News Article Clusters Visualized via PCA (2D Projection)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(title='Cluster / Actual Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()